In [105]:
#data loading
import pandas as pd

X_train = pd.read_csv("../data/processed/X_train.csv")
y_train = pd.read_csv("../data/processed/y_train.csv").squeeze()

X_val = pd.read_csv("../data/processed/X_val.csv")
y_val = pd.read_csv("../data/processed/y_val.csv").squeeze()
X_test = pd.read_csv("../data/processed/X_test.csv")
y_test = pd.read_csv("../data/processed/y_test.csv").squeeze()

In [106]:
print(X_train.shape)
print(y_train.shape)

(1980643, 78)
(1980643,)


In [107]:
negative_duration = X_train[X_train['Flow Duration']<0]
print(negative_duration.shape)
#go into y_train and fetch only the labels at those exact row positions.
print(y_train[negative_duration.index].value_counts())

(75, 78)
Label
BENIGN    75
Name: count, dtype: int64


In [108]:
#all negative flow duration are benign so drop it
bad_idx = negative_duration.index
X_train = X_train.drop(index= bad_idx)
y_train = y_train.drop(index= bad_idx)

print(X_train.shape)
print(y_train.shape)

(1980568, 78)
(1980568,)


In [109]:
sparse_flags = ['Fwd PSH Flags', 'Bwd PSH Flags', 'Fwd URG Flags', 'Bwd URG Flags']
print(X_train[sparse_flags].value_counts().head(10))

Fwd PSH Flags  Bwd PSH Flags  Fwd URG Flags  Bwd URG Flags
0              0              0              0                1888547
1              0              0              0                  91798
                              1              0                    223
Name: count, dtype: int64


In [110]:
#dropping all four flags because all of it contains 0(mostly)
urg_rows = X_train[X_train['Fwd URG Flags'] >0]
print(y_train[urg_rows.index].value_counts())

Label
BENIGN    223
Name: count, dtype: int64


In [111]:
sparse_flags = ['Fwd PSH Flags', 'Bwd PSH Flags', 'Fwd URG Flags', 'Bwd URG Flags']
X_train = X_train.drop(columns=sparse_flags)
print(X_train.shape)

(1980568, 74)


In [112]:
sparse_flags2 = ['RST Flag Count', 'CWE Flag Count', 'ECE Flag Count']
X_train = X_train.drop(columns=sparse_flags2)
X_val = X_val.drop(columns=sparse_flags2)
X_test = X_test.drop(columns=sparse_flags2)

print(X_train.shape)

(1980568, 71)


In [113]:
X_val = X_val.drop(columns=sparse_flags)
X_test = X_test.drop(columns=sparse_flags)

print(X_val.shape)
print(X_test.shape)

(423051, 71)
(424182, 71)


In [114]:
print(y_train.value_counts())

Label
BENIGN                        1590757
DoS Hulk                       161178
PortScan                       111226
DDoS                            89668
DoS GoldenEye                    7209
FTP-Patator                      5558
SSH-Patator                      4130
DoS slowloris                    4060
DoS Slowhttptest                 3851
Bot                              1370
Web Attack � Brute Force         1056
Web Attack � XSS                  457
Infiltration                       26
Web Attack � Sql Injection         15
Heartbleed                          7
Name: count, dtype: int64


In [115]:
print(y_train.unique())


<StringArray>
[                    'BENIGN',                   'DoS Hulk',
                   'PortScan',                       'DDoS',
           'DoS Slowhttptest',              'DoS GoldenEye',
                'SSH-Patator',                'FTP-Patator',
               'Infiltration',              'DoS slowloris',
                        'Bot',                 'Heartbleed',
   'Web Attack � Brute Force',           'Web Attack � XSS',
 'Web Attack � Sql Injection']
Length: 15, dtype: str


In [116]:
#unicode fix
y_train = y_train.str.replace('Web Attack \ufffd', 'Web Attack -', regex=False)
y_val = y_val.str.replace('Web Attack \ufffd', 'Web Attack -', regex=False)
y_test = y_test.str.replace('Web Attack \ufffd', 'Web Attack -', regex=False)

print(y_train.unique())

<StringArray>
[                    'BENIGN',                   'DoS Hulk',
                   'PortScan',                       'DDoS',
           'DoS Slowhttptest',              'DoS GoldenEye',
                'SSH-Patator',                'FTP-Patator',
               'Infiltration',              'DoS slowloris',
                        'Bot',                 'Heartbleed',
   'Web Attack - Brute Force',           'Web Attack - XSS',
 'Web Attack - Sql Injection']
Length: 15, dtype: str


In [117]:
print(type(y_train))
print(type(y_val))
print(type(y_test))

<class 'pandas.Series'>
<class 'pandas.Series'>
<class 'pandas.Series'>


In [118]:
# import numpy as np

# np.save('../data/processed/X_train_resampled.npy', X_train_resampled)
# y_train_resampled.to_csv('../data/processed/y_train_resampled.csv', index=False)

# print("Saved")

In [119]:
# np.save('../data/processed/X_val_scaled.npy', X_val_scaled)
# np.save('../data/processed/X_test_scaled.npy', X_test_scaled)

# print("Saved.")

In [120]:
#Standard Scaler to normalize

from sklearn.preprocessing import StandardScaler
import joblib

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

joblib.dump(scaler, '../models/standard_scaler.pkl')
print("Scaler saved.")
print("X_train_scaled shape:", X_train_scaled.shape)

Scaler saved.
X_train_scaled shape: (1980568, 71)


In [121]:
#for random forest and xgboost
X_train.to_csv('../data/processed/X_train_clean.csv', index=False)
X_val.to_csv('../data/processed/X_val_clean.csv', index=False)
X_test.to_csv('../data/processed/X_test_clean.csv', index=False)
y_train.to_csv('../data/processed/y_train_clean.csv', index=False)
y_val.to_csv('../data/processed/y_val_clean.csv', index=False)
y_test.to_csv('../data/processed/y_test_clean.csv', index=False)

print("Clean unscaled data saved.")

Clean unscaled data saved.


In [122]:
#SMOTE
from imblearn.over_sampling import SMOTE

sampling_strategy = {
    'DoS GoldenEye': 10000,
    'FTP-Patator': 10000,
    'SSH-Patator': 10000,
    'DoS slowloris': 10000,
    'DoS Slowhttptest': 10000,
    'Bot': 10000,
    'Web Attack - Brute Force': 10000,
    'Web Attack - XSS': 10000,
    'Infiltration': 10000,
    'Web Attack - Sql Injection': 10000,
    'Heartbleed': 10000
}

smote = SMOTE(sampling_strategy=sampling_strategy, random_state=42)
X_train_resampled, y_train_resampled = smote.fit_resample(X_train_scaled, y_train)

print(y_train_resampled.value_counts())

Label
BENIGN                        1590757
DoS Hulk                       161178
PortScan                       111226
DDoS                            89668
DoS Slowhttptest                10000
DoS GoldenEye                   10000
SSH-Patator                     10000
FTP-Patator                     10000
Infiltration                    10000
DoS slowloris                   10000
Bot                             10000
Heartbleed                      10000
Web Attack - Brute Force        10000
Web Attack - XSS                10000
Web Attack - Sql Injection      10000
Name: count, dtype: int64


In [123]:
#for deep learning
import numpy as np

np.save('../data/processed/X_train_resampled.npy', X_train_resampled)
np.save('../data/processed/X_val_scaled.npy', X_val_scaled)
np.save('../data/processed/X_test_scaled.npy', X_test_scaled)

y_train_resampled.to_csv('../data/processed/y_train_resampled.csv', index=False)

print("All saved.")

All saved.
